# Frozen Lake Example

In [31]:
import gymnasium as gym
import numpy as np
import pygame
from techdays26.frozen_lake.frozen_lake_utils import (
    get_action_from_keyboard,
    key_to_action,
    action_to_arrow,
)
from techdays26.frozen_lake.frozen_lake_enhanced import generate_random_map

## Registering the Frozen Lake environment

In [32]:
gym.register(
    id="FrozenLake-v2",
    entry_point="techdays26.frozen_lake.frozen_lake_enhanced:FrozenLakeEnv",
    kwargs={"map_name": "8x8"},
    max_episode_steps=200,
    reward_threshold=0.85,
)

/home/xcx5972/github/techdays26/venv/lib/python3.12/site-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment FrozenLake-v2 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


## Introducing the Problem

In [ ]:
# https://github.com/johnnycode8/gym_solutions

import numpy as np
import pygame
from techdays26.frozen_lake.frozen_lake_utils import (
    get_action_from_keyboard,
    key_to_action,
    action_to_arrow,
)
from techdays26.frozen_lake.frozen_lake_enhanced import generate_random_map

episodes: int = 2

env = gym.make(
    "FrozenLake-v2",
    desc=None,  # generate_random_map(size=8)
    map_name="8x8",
    show_q_labels=False,
    is_slippery=False,
    success_rate=1.0,
    reward_schedule=(1, -1, 0),
    render_mode="human",
)

try:
    env.unwrapped.set_info({
        "Mode": "Manual (Arrow Keys)",
    })

    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        while not terminated and not truncated:
            env.unwrapped.set_episode(i)

            # Wait for an arrow key press
            action = get_action_from_keyboard(key_to_action)
            if action is None:
                raise SystemExit  # on Escape

            new_state, reward, terminated, truncated, info_dict = env.step(action)

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

        # Brief pause after episode ends so the player can see the result
        pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

ESC button pressed. Done!


## The Policy Function
Illustrate how $\pi(s,a)$ works

In [34]:
from techdays26.frozen_lake.frozen_lake_utils import action_to_arrow

print("Mapping of actions to directions:")
print(action_to_arrow)

Mapping of actions to directions:
{0: '←', 1: '↓ ', 2: '→', 3: '↑ '}


### Specify a simple Policy
What does the following policies do? Can you guess?

In [35]:
# To act, sample from the distribution:
def act(state, π):  # TODO: typing
    probs = [π(state, a) for a in range(4)]
    return np.random.choice([0, 1, 2, 3], p=probs)

In [36]:
def π_1(state, action):
    """π(a|s) → probability"""
    #         ←    ↓    →    ↑
    probs = [0.05, 0.45, 0.45, 0.05]
    return probs[action]

In [ ]:
def π_2(state: int, action: int):
    """π(a|s) → probability"""
    if state % 8 < 7:
        #         ←    ↓    →    ↑
        probs = [0.0, 0.0, 1.0, 0.0]
    else:
        #         ←    ↓    →    ↑
        probs = [0.0, 1.0, 0.0, 0.0]

    return probs[action]

### Running an agent with above policies

In [39]:
# https://github.com/johnnycode8/gym_solutions

episodes: int = 20
π = π_1

env = gym.make(
    "FrozenLake-v2",
    desc=None,  # generate_random_map(size=8)
    map_name="8x8",
    show_q_labels=False,
    is_slippery=False,
    success_rate=1.0,
    reward_schedule=(1, -1, 0),
    render_mode="human",
)

try:
    env.unwrapped.set_info({
        "Mode": "Manual (Arrow Keys)",
    })

    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        while not terminated and not truncated:
            env.unwrapped.set_episode(i)

            ####################################################################
            action = act(state=state, π=π)
            ####################################################################

            new_state, reward, terminated, truncated, info_dict = env.step(action)

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

            state = new_state

        # Brief pause after episode ends so the player can see the result
        pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

ESC button pressed. Done!
